In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

In [2]:
df = pd.read_csv('sleep.csv')

In [3]:
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea



Столбцы набора данных:  
- ID человека: идентификатор для каждого человека.  
- Пол: пол человека (мужчина/женщина).  
- Возраст: возраст человека в годах.  
- Род занятий: род занятий или профессия человека.  
- Продолжительность сна (часы): количество часов, которые человек спит в день.  
- Качество сна (шкала: 1-10): субъективная оценка качества сна в диапазоне от 1 до 10.  
- Уровень физической активности (минут/день): количество минут, в течение которых человек занимается физической активностью ежедневно.  
- Уровень стресса (шкала: 1-10): субъективная оценка уровня стресса, испытываемого человеком, в диапазоне от 1 до 10.  
- Категория ИМТ: категория индекса массы тела человека (например, недостаточный вес, нормальный вес, избыточный вес).  
- Артериальное давление (систолическое/диастолическое): измерение артериального давления человека, указанное как систолическое давление над диастолическим давлением.  
- Частота сердечных сокращений (уд/мин): частота сердечных сокращений человека в состоянии покоя в ударах в минуту.  
- Ежедневные шаги: количество шагов, которые человек делает за день.  
- Нарушение сна: наличие или отсутствие нарушения сна у человека (отсутствует, бессонница, апноэ во сне).  


### **Выделим и предобработаем целевую переменную: нарушение сна**

In [4]:
df['Sleep Disorder'].unique()

array([nan, 'Sleep Apnea', 'Insomnia'], dtype=object)

**Заменим nan на 'doesn't have'**

In [5]:
df['Sleep Disorder'] = df['Sleep Disorder'].fillna("doesn't have")
df['Sleep Disorder'].unique()

array(["doesn't have", 'Sleep Apnea', 'Insomnia'], dtype=object)

In [6]:
sleep_dic = {'Insomnia': 0, "doesn't have":1, 'Sleep Apnea':2}
df['Sleep Disorder'] = df['Sleep Disorder'].map(sleep_dic)

In [7]:
Y = df['Sleep Disorder']
df = df.drop('Sleep Disorder', axis=1)

### **Предобработаю данные**

**Удалю незначащую колокнку с айди и преобразую гендер к бинарному числовому, а не текстовому виду**

In [8]:
df = df.drop('Person ID', axis=1)

In [9]:
df['Gender'] = df['Gender'].map({"Male":0, "Female":1})

In [10]:
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps
0,0,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200
1,0,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000
2,0,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000
3,0,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000
4,0,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000


**Разобью возраст на категории по 5 лет, тк идейно, что 23, что 24. Разницы большой нет (влияет не возраст в пределах 5 лет, а показатели организма)** 

In [11]:
print(df['Age'].min(), df['Age'].max())

27 59


In [12]:
groups = []
for i in range(df['Age'].min() - 1, df['Age'].max() + 5, 5):
    groups.append(i)
groups_cut = pd.cut(df['Age'].values, groups)
df['Age'] = groups_cut.codes

In [13]:
print(df['Age'].min(), df['Age'].max())

0 6


In [14]:
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps
0,0,0,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200
1,0,0,Doctor,6.2,6,60,8,Normal,125/80,75,10000
2,0,0,Doctor,6.2,6,60,8,Normal,125/80,75,10000
3,0,0,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000
4,0,0,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000


In [15]:
df['Occupation'].unique()

array(['Software Engineer', 'Doctor', 'Sales Representative', 'Teacher',
       'Nurse', 'Engineer', 'Accountant', 'Scientist', 'Lawyer',
       'Salesperson', 'Manager'], dtype=object)

**Нельзя как-то эмпирически разделить эти профессии однозначно, поэтому воспользуюсь просто лейбл декодером**

In [16]:

le = LabelEncoder()
le.fit(df['Occupation'].unique())
le.classes_

array(['Accountant', 'Doctor', 'Engineer', 'Lawyer', 'Manager', 'Nurse',
       'Sales Representative', 'Salesperson', 'Scientist',
       'Software Engineer', 'Teacher'], dtype=object)

In [17]:
df['Occupation'] = le.transform(df['Occupation'])

In [18]:
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps
0,0,0,9,6.1,6,42,6,Overweight,126/83,77,4200
1,0,0,1,6.2,6,60,8,Normal,125/80,75,10000
2,0,0,1,6.2,6,60,8,Normal,125/80,75,10000
3,0,0,6,5.9,4,30,8,Obese,140/90,85,3000
4,0,0,6,5.9,4,30,8,Obese,140/90,85,3000


**Разобьем сон на группы с ходом 0.5 (фазы сна)**

In [19]:
print(df['Sleep Duration'].min(), df['Sleep Duration'].max())

5.8 8.5


In [20]:
df['Sleep Duration'] = df['Sleep Duration'] * 10

In [21]:
groups = []
for i in range(54, 90, 5):
    groups.append(i)
groups_cut = pd.cut(df['Sleep Duration'].values, groups)
df['Sleep Duration'] = groups_cut.codes

In [22]:
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps
0,0,0,9,1,6,42,6,Overweight,126/83,77,4200
1,0,0,1,1,6,60,8,Normal,125/80,75,10000
2,0,0,1,1,6,60,8,Normal,125/80,75,10000
3,0,0,6,0,4,30,8,Obese,140/90,85,3000
4,0,0,6,0,4,30,8,Obese,140/90,85,3000


**Закодирую Категория ИМТ**

In [23]:
le2 = LabelEncoder()
le2.fit(df['BMI Category'].unique())
df['BMI Category'] = le2.transform(df['BMI Category'])

In [24]:
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps
0,0,0,9,1,6,42,6,3,126/83,77,4200
1,0,0,1,1,6,60,8,0,125/80,75,10000
2,0,0,1,1,6,60,8,0,125/80,75,10000
3,0,0,6,0,4,30,8,2,140/90,85,3000
4,0,0,6,0,4,30,8,2,140/90,85,3000


**Предобработаю давление**

In [25]:
df[['Systolic', 'Diastolic']] = df['Blood Pressure'].str.split('/', expand=True)

In [26]:
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Systolic,Diastolic
0,0,0,9,1,6,42,6,3,126/83,77,4200,126,83
1,0,0,1,1,6,60,8,0,125/80,75,10000,125,80
2,0,0,1,1,6,60,8,0,125/80,75,10000,125,80
3,0,0,6,0,4,30,8,2,140/90,85,3000,140,90
4,0,0,6,0,4,30,8,2,140/90,85,3000,140,90


In [27]:
df['Systolic'] = df['Systolic'].astype(int)
df['Diastolic'] = df['Diastolic'].astype(int)

In [28]:
def determine_pressure_status(row):
    systolic = row['Systolic']
    diastolic = row['Diastolic']
    if systolic < 90 or diastolic < 60:
        return -1  # Пониженное давление
    elif systolic > 120 or diastolic > 80:
        return 1   # Повышенное давление
    else:
        return 0   # Нормальное давление
df['PressureStatus'] = df.apply(determine_pressure_status, axis=1)

In [29]:
df = df.drop(['Blood Pressure', 'Systolic', 'Diastolic'], axis=1)


In [30]:
df['Daily Steps'].unique()

array([ 4200, 10000,  3000,  3500,  8000,  4000,  4100,  6800,  5000,
        7000,  5500,  5200,  5600,  3300,  4800,  7500,  7300,  6200,
        6000,  3700])

In [31]:
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,PressureStatus
0,0,0,9,1,6,42,6,3,77,4200,1
1,0,0,1,1,6,60,8,0,75,10000,1
2,0,0,1,1,6,60,8,0,75,10000,1
3,0,0,6,0,4,30,8,2,85,3000,1
4,0,0,6,0,4,30,8,2,85,3000,1


**Проверим на наличие пропусков в данных**

In [32]:
df.isna().sum()

Gender                     0
Age                        0
Occupation                 0
Sleep Duration             0
Quality of Sleep           0
Physical Activity Level    0
Stress Level               0
BMI Category               0
Heart Rate                 0
Daily Steps                0
PressureStatus             0
dtype: int64

### **Разобью данные на тренировочные и тестовые**

In [131]:
x_train, x_test, y_train, y_test = train_test_split(df, Y, random_state=100, test_size=0.3, stratify=Y)

## **Для полученных моделей с высоким качеством (2-3 модели) выполнить кросс-валидацию и сравнить результаты.**

### **Голосование большинством**

In [132]:
from sklearn.ensemble import VotingClassifier
from sklearn.compose import ColumnTransformer, make_column_selector

In [133]:
scale_features = ["Physical Activity Level", "Heart Rate", "Daily Steps", "Quality of Sleep", "Sleep Duration"]
preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), scale_features)
], remainder='passthrough')  # Оставляем остальные признаки без изменений'
pipeline1 = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(random_state=100, max_iter=1000))
])
pipeline2 = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=7)) 
])
pipeline3 = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=6, random_state=42)) 
])
eclf = VotingClassifier(estimators=[('lr', pipeline1), ('knn', pipeline2), ('dt', pipeline3)], voting='hard')
eclf.fit(x_train, y_train)
train_predict = eclf.predict(x_train)
test_predict = eclf.predict(x_test)

In [134]:
print(f1_score(y_train, train_predict, average = 'macro'))
print(f1_score(y_test, test_predict, average = 'macro'))

0.9014482615551939
0.8523293270532185


In [135]:
from sklearn.model_selection import cross_val_score
model1 = VotingClassifier(estimators=[('lr', pipeline1), ('knn', pipeline2), ('dt', pipeline3)], voting='hard')
scores = cross_val_score(model1, x_train, y_train, cv=5, scoring='f1_macro')
print(scores)
print(scores.mean())

[0.81748191 0.84448653 0.8194855  0.93042494 0.89313747]
0.8610032705015385


### **Случайный лес**

In [136]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV

clf = RandomForestClassifier(random_state=100, n_estimators=7, max_features=6)
scores = cross_val_score(clf, x_train, y_train, cv=5, scoring='f1_macro')
print(scores)
print(scores.mean())

[0.80185645 0.84448653 0.83736406 0.93042494 0.86880607]
0.8565876103709039


In [137]:
clf.fit(x_train, y_train)
train_predict = clf.predict(x_train)
test_predict = clf.predict(x_test)

print(f1_score(y_train, train_predict, average='macro'))
print(f1_score(y_test, test_predict, average='macro'))

0.9229998841541572
0.9018005110203692


### **Бустинг**

In [138]:
from sklearn.ensemble import GradientBoostingClassifier

clf = GradientBoostingClassifier(n_estimators=25)

scores = cross_val_score(clf, x_train, y_train, cv=5, scoring='f1_macro')
print(scores)
print(scores.mean())



[0.82697947 0.82091982 0.88686869 0.93042494 0.88634838]
0.8703082607195316


In [139]:
clf.fit(x_train, y_train)
train_predict = clf.predict(x_train)
test_predict = clf.predict(x_test)
print(f1_score(y_train, train_predict, average='macro'))
print(f1_score(y_test, test_predict, average='macro'))

0.9229998841541572
0.89297829036635


**Наилучший средний f-score на фолдах у бустинга**

## Разбить исходную выборку на 3 подвыборки (обучающая, валидационная, тестовая). Для полученных моделей с высоким качеством (2-3 модели) определить наилучшие комбинации параметров, дающие лучшее значение по метрике F1 и сравнить результаты.

In [140]:
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.3, random_state=100, stratify=y_train)

In [141]:
pip install optuna


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Голосование большинством

In [142]:
def objective_lr(trial):
    C = trial.suggest_loguniform('classifier__C', 0.01, 100)
    max_iter = trial.suggest_categorical('classifier__max_iter', [1000, 5000])
    pipeline1.set_params(classifier__C=C, classifier__max_iter=max_iter)
    return cross_val_score(pipeline1, x_train, y_train, cv=5, scoring='f1_macro').mean()

def objective_knn(trial):
    n_neighbors = trial.suggest_int('classifier__n_neighbors', 2, 10)
    pipeline2.set_params(classifier__n_neighbors=n_neighbors)
    return cross_val_score(pipeline2, x_train, y_train, cv=5, scoring='f1_macro').mean()

def objective_dt(trial):
    max_depth = trial.suggest_int('classifier__max_depth', 5, 9)
    pipeline3.set_params(classifier__max_depth=max_depth)
    return cross_val_score(pipeline3, x_train, y_train, cv=5, scoring='f1_macro').mean()

study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=20)

study_knn = optuna.create_study(direction='maximize')
study_knn.optimize(objective_knn, n_trials=20)

study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(objective_dt, n_trials=20)

best_lr = pipeline1.set_params(**study_lr.best_params).fit(x_train, y_train)
best_knn = pipeline2.set_params(**study_knn.best_params).fit(x_train, y_train)
best_dt = pipeline3.set_params(**study_dt.best_params).fit(x_train, y_train)

eclf = VotingClassifier(estimators=[('lr', best_lr), ('knn', best_knn), ('dt', best_dt)], voting='hard')
eclf.fit(x_train, y_train)

val_predict = eclf.predict(x_val)
test_predict = eclf.predict(x_test)

print(f'F1 на валидационной {f1_score(y_val, val_predict, average="macro")}')
print(f'F1 на тестовой {f1_score(y_test, test_predict, average="macro")}')

[I 2025-03-16 16:38:49,594] A new study created in memory with name: no-name-33d9ee59-cb99-4f69-9d0f-8c6690ba0865
C:\Users\User\AppData\Local\Temp\ipykernel_21536\1337643051.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('classifier__C', 0.01, 100)
[I 2025-03-16 16:38:49,751] Trial 0 finished with value: 0.8893218991577404 and parameters: {'classifier__C': 3.789797915559348, 'classifier__max_iter': 1000}. Best is trial 0 with value: 0.8893218991577404.
C:\Users\User\AppData\Local\Temp\ipykernel_21536\1337643051.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('classifier__C', 0.01, 100)
[I 2025-03-16 16:3

F1 на валидационной 0.8624542124542125
F1 на тестовой 0.860617039964866


#### Случайный лес

In [143]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 1, 13)
    max_features = trial.suggest_int('max_features', 1, 10)
    
    clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, random_state=100)
    score = cross_val_score(clf, x_train, y_train, cv=5, scoring='f1_macro').mean()
    
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

best_rf = RandomForestClassifier(**study.best_params, random_state=100)
best_rf.fit(x_train, y_train)

[I 2025-03-16 16:38:54,653] A new study created in memory with name: no-name-3a5018f7-4f70-4319-9c3c-3a3321985e6c
[I 2025-03-16 16:38:54,746] Trial 0 finished with value: 0.8389549688594601 and parameters: {'n_estimators': 11, 'max_features': 1}. Best is trial 0 with value: 0.8389549688594601.
[I 2025-03-16 16:38:54,791] Trial 1 finished with value: 0.8336205637126058 and parameters: {'n_estimators': 5, 'max_features': 10}. Best is trial 0 with value: 0.8389549688594601.
[I 2025-03-16 16:38:54,840] Trial 2 finished with value: 0.813820994675941 and parameters: {'n_estimators': 7, 'max_features': 5}. Best is trial 0 with value: 0.8389549688594601.
[I 2025-03-16 16:38:54,868] Trial 3 finished with value: 0.7995277779205374 and parameters: {'n_estimators': 1, 'max_features': 10}. Best is trial 0 with value: 0.8389549688594601.
[I 2025-03-16 16:38:54,911] Trial 4 finished with value: 0.8224974985972429 and parameters: {'n_estimators': 4, 'max_features': 6}. Best is trial 0 with value: 0.83

RandomForestClassifier(max_features=2, n_estimators=2, random_state=100)

In [144]:
val_predict = best_rf.predict(x_val)
test_predict = best_rf.predict(x_test)

print(f'F1 на валидационной {f1_score(y_val, val_predict, average='macro')}')
print(f'F1 на тестовой {f1_score(y_test, test_predict, average='macro')}')

F1 на валидационной 0.826103779732812
F1 на тестовой 0.8477256153042033


#### Бустинг

In [145]:
def objective(trial):
    n_estimators = trial.suggest_categorical('n_estimators', [10, 20, 50, 100])
    learning_rate = trial.suggest_categorical('learning_rate', [0.01, 0.05, 0.1, 0.2])
    max_depth = trial.suggest_categorical('max_depth', [3, 5, 7, 10])
    
    clf = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth)
    score = cross_val_score(clf, x_train, y_train, cv=5, scoring='f1_macro').mean()
    
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

best_boost = GradientBoostingClassifier(**study.best_params)
best_boost.fit(x_train, y_train)

y_test_pred = best_boost.predict(x_test)
f1_test = f1_score(y_test, y_test_pred, average='macro')

[I 2025-03-16 16:38:55,771] A new study created in memory with name: no-name-ea372208-6424-4e97-9577-9b40f69b4aab
[I 2025-03-16 16:38:55,974] Trial 0 finished with value: 0.8255795176573741 and parameters: {'n_estimators': 10, 'learning_rate': 0.2, 'max_depth': 10}. Best is trial 0 with value: 0.8255795176573741.
[I 2025-03-16 16:38:56,347] Trial 1 finished with value: 0.8340410561189125 and parameters: {'n_estimators': 20, 'learning_rate': 0.05, 'max_depth': 7}. Best is trial 1 with value: 0.8340410561189125.
[I 2025-03-16 16:38:57,130] Trial 2 finished with value: 0.8405834026612592 and parameters: {'n_estimators': 50, 'learning_rate': 0.05, 'max_depth': 10}. Best is trial 2 with value: 0.8405834026612592.
[I 2025-03-16 16:38:58,021] Trial 3 finished with value: 0.8405834026612592 and parameters: {'n_estimators': 50, 'learning_rate': 0.05, 'max_depth': 10}. Best is trial 2 with value: 0.8405834026612592.
[I 2025-03-16 16:38:58,245] Trial 4 finished with value: 0.8324109905688853 and 

In [146]:
val_predict = best_boost.predict(x_val)
test_predict = best_boost.predict(x_test)

print(f'F1 на валидационной {f1_score(y_val, val_predict, average='macro')}')
print(f'F1 на тестовой {f1_score(y_test, test_predict, average='macro')}')

F1 на валидационной 0.8557118499573743
F1 на тестовой 0.8763078353871192


### Подготовить и обучить итоговую модель (ансамбль) с применением различных техник. Выполнить сравнительный анализ полученных результатов.

In [83]:
x_train = pd.concat([x_train, x_val], axis=0)
y_train = pd.concat([y_train, y_val], axis=0)

In [91]:
from sklearn.ensemble import StackingClassifier

In [101]:
def objective_lr(trial):
    C = trial.suggest_loguniform('classifier__C', 0.01, 100)
    max_iter = trial.suggest_categorical('classifier__max_iter', [1000, 5000])
    pipeline1.set_params(classifier__C=C, classifier__max_iter=max_iter)
    return cross_val_score(pipeline1, x_train, y_train, cv=5, scoring='f1_macro').mean()

study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=20)
best_lr = pipeline1.set_params(**study_lr.best_params).fit(x_train, y_train)

# Оптимизация гиперпараметров для KNN
def objective_knn(trial):
    n_neighbors = trial.suggest_int('classifier__n_neighbors', 2, 10)
    pipeline2.set_params(classifier__n_neighbors=n_neighbors)
    return cross_val_score(pipeline2, x_train, y_train, cv=5, scoring='f1_macro').mean()

study_knn = optuna.create_study(direction='maximize')
study_knn.optimize(objective_knn, n_trials=20)
best_knn = pipeline2.set_params(**study_knn.best_params).fit(x_train, y_train)

# Оптимизация гиперпараметров для дерева решений
def objective_dt(trial):
    max_depth = trial.suggest_int('classifier__max_depth', 5, 9)
    pipeline3.set_params(classifier__max_depth=max_depth)
    return cross_val_score(pipeline3, x_train, y_train, cv=5, scoring='f1_macro').mean()

study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(objective_dt, n_trials=20)
best_dt = pipeline3.set_params(**study_dt.best_params).fit(x_train, y_train)

# Создание вотинг-классификатора
voting_clf = VotingClassifier(estimators=[
    ('lr', best_lr),
    ('knn', best_knn),
    ('dt', best_dt)
], voting='hard')

# Обучение случайного леса
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 1, 13)
    max_features = trial.suggest_int('max_features', 1, 10)
    
    clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, random_state=100)
    return cross_val_score(clf, x_train, y_train, cv=5, scoring='f1_macro').mean()

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=20)
best_rf = RandomForestClassifier(**study_rf.best_params, random_state=100).fit(x_train, y_train)

# Оптимизация градиентного бустинга
def objective_boost(trial):
    n_estimators = trial.suggest_categorical('n_estimators', [10, 20, 50, 100])
    learning_rate = trial.suggest_categorical('learning_rate', [0.01, 0.05, 0.1, 0.2])
    max_depth = trial.suggest_categorical('max_depth', [3, 5, 7, 10])
    
    clf = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth)
    return cross_val_score(clf, x_train, y_train, cv=5, scoring='f1_macro').mean()

study_boost = optuna.create_study(direction='maximize')
study_boost.optimize(objective_boost, n_trials=20)
best_boost = GradientBoostingClassifier(**study_boost.best_params).fit(x_train, y_train)

# Создание стекинг-классификатора, где вотинг-классификатор, случайный лес и градиентный бустинг - базовые модели
stacking_clf = StackingClassifier(estimators=[
    ('voting', voting_clf),
    ('rf', best_rf),
    ('boost', best_boost)
], final_estimator=LogisticRegression())

# Обучение стекинга
stacking_clf.fit(x_train, y_train)

[I 2025-03-16 16:27:04,992] A new study created in memory with name: no-name-4b261dc7-4d4d-420c-b5b9-03b7563013a4
C:\Users\User\AppData\Local\Temp\ipykernel_21536\2126275131.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('classifier__C', 0.01, 100)
[I 2025-03-16 16:27:05,034] Trial 0 finished with value: 0.8599006651147354 and parameters: {'classifier__C': 0.05068848054751701, 'classifier__max_iter': 1000}. Best is trial 0 with value: 0.8599006651147354.
C:\Users\User\AppData\Local\Temp\ipykernel_21536\2126275131.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('classifier__C', 0.01, 100)
[I 2025-03-16 16

StackingClassifier(estimators=[('voting',
                                VotingClassifier(estimators=[('lr',
                                                              Pipeline(steps=[('scaler',
                                                                               StandardScaler()),
                                                                              ('classifier',
                                                                               LogisticRegression(C=34.7683336314779,
                                                                                                  max_iter=1000))])),
                                                             ('knn',
                                                              Pipeline(steps=[('scaler',
                                                                               StandardScaler()),
                                                                              ('classifier',
                                                                               KNeighborsClassifier(n_neighbors=4))])),
                                                             ('dt',
                                                              Pipeline(steps=[('scaler',
                                                                               StandardScaler()),
                                                                              ('classifier',
                                                                               DecisionTreeClassifier(max_depth=5))]))])),
                               ('rf',
                                RandomForestClassifier(max_features=4,
                                                       n_estimators=2,
                                                       random_state=100)),
                               ('boost',
                                GradientBoostingClassifier(n_estimators=20))],
                   final_estimator=LogisticRegression())

In [102]:
train_predict = stacking_clf.predict(x_train)
test_predict = stacking_clf.predict(x_test)

print("F1-score (train):", f1_score(y_train, train_predict, average='macro'))
print("F1-score (test):", f1_score(y_test, test_predict, average='macro'))


F1-score (train): 0.9168423988342181
F1-score (test): 0.8724645987057335


In [99]:
def objective_lr(trial):
    C = trial.suggest_loguniform('classifier__C', 0.01, 100)
    max_iter = trial.suggest_categorical('classifier__max_iter', [1000, 5000])
    pipeline1.set_params(classifier__C=C, classifier__max_iter=max_iter)
    return cross_val_score(pipeline1, x_train, y_train, cv=5, scoring='f1_macro').mean()

# Для второй модели (KNN)
def objective_knn(trial):
    n_neighbors = trial.suggest_int('classifier__n_neighbors', 2, 10)
    pipeline2.set_params(classifier__n_neighbors=n_neighbors)
    return cross_val_score(pipeline2, x_train, y_train, cv=5, scoring='f1_macro').mean()

# Для третьей модели (дерево решений)
def objective_dt(trial):
    max_depth = trial.suggest_int('classifier__max_depth', 5, 9)
    pipeline3.set_params(classifier__max_depth=max_depth)
    return cross_val_score(pipeline3, x_train, y_train, cv=5, scoring='f1_macro').mean()

# Оптимизация гиперпараметров для каждой модели
study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=20)

study_knn = optuna.create_study(direction='maximize')
study_knn.optimize(objective_knn, n_trials=20)

study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(objective_dt, n_trials=20)

# Обучение моделей с лучшими параметрами
best_lr = pipeline1.set_params(**study_lr.best_params).fit(x_train, y_train)
best_knn = pipeline2.set_params(**study_knn.best_params).fit(x_train, y_train)
best_dt = pipeline3.set_params(**study_dt.best_params).fit(x_train, y_train)

# Обучение случайного леса
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 1, 13)
    max_features = trial.suggest_int('max_features', 1, 10)
    
    clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, random_state=100)
    score = cross_val_score(clf, x_train, y_train, cv=5, scoring='f1_macro').mean()
    
    return score

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=20)

best_rf = RandomForestClassifier(**study_rf.best_params, random_state=100).fit(x_train, y_train)

# Обучение модели градиентного бустинга
def objective_boost(trial):
    n_estimators = trial.suggest_categorical('n_estimators', [10, 20, 50, 100])
    learning_rate = trial.suggest_categorical('learning_rate', [0.01, 0.05, 0.1, 0.2])
    max_depth = trial.suggest_categorical('max_depth', [3, 5, 7, 10])
    
    clf = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth)
    score = cross_val_score(clf, x_train, y_train, cv=5, scoring='f1_macro').mean()
    
    return score

study_boost = optuna.create_study(direction='maximize')
study_boost.optimize(objective_boost, n_trials=20)

best_boost = GradientBoostingClassifier(**study_boost.best_params).fit(x_train, y_train)

# Создание стекинг-ассембля с тремя базовыми моделями
estimators = [
    ('lr', best_lr),
    ('knn', best_knn),
    ('dt', best_dt),
    ('rf', best_rf),
    ('boost', best_boost)
]

stacking_clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())

# Обучение стекинг-ассембля
stacking_clf.fit(x_train, y_train)



[I 2025-03-16 16:22:27,461] A new study created in memory with name: no-name-29a842ac-ce7b-417e-81a5-c1dc524e1eb9
C:\Users\User\AppData\Local\Temp\ipykernel_21536\1743787916.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('classifier__C', 0.01, 100)
[I 2025-03-16 16:22:27,506] Trial 0 finished with value: 0.8739177095075361 and parameters: {'classifier__C': 5.551792776536298, 'classifier__max_iter': 1000}. Best is trial 0 with value: 0.8739177095075361.
C:\Users\User\AppData\Local\Temp\ipykernel_21536\1743787916.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('classifier__C', 0.01, 100)
[I 2025-03-16 16:2

StackingClassifier(estimators=[('lr',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('classifier',
                                                 LogisticRegression(C=5.551792776536298,
                                                                    max_iter=1000))])),
                               ('knn',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('classifier',
                                                 KNeighborsClassifier(n_neighbors=4))])),
                               ('dt',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('classifier',
                                                 DecisionTreeClassifier(max_depth=5))])),
                               ('rf',
                                RandomForestClassifier(max_features=4,
                                                       n_estimators=4,
                                                       random_state=100)),
                               ('boost',
                                GradientBoostingClassifier(learning_rate=0.2,
                                                           n_estimators=20))],
                   final_estimator=LogisticRegression())

In [100]:
train_predict = stacking_clf.predict(x_train)
test_predict = stacking_clf.predict(x_test)

print("F1-score (train):", f1_score(y_train, train_predict, average='macro'))
print("F1-score (test):", f1_score(y_test, test_predict, average='macro'))

F1-score (train): 0.9229998841541572
F1-score (test): 0.9018005110203692
